<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-04-image-and-multimodal-generation/lesson-4.3-controlled-generation/notebooks/exercises-4.3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4.3 — Controlled Generation
Netsetos GenAI Course v2.0 | Module 4

ControlNet, inpainting, multi-ControlNet, post-processing, regional prompting, India use cases


In [ ]:
# Setup
import torch
print(f'CUDA: {torch.cuda.is_available()}')
# pip install diffusers transformers accelerate -q


## Ex 1: ControlNet Conditioning Scale


In [ ]:
# from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline
# import cv2
# from PIL import Image

# Extract canny edges
# img = cv2.imread('building.jpg')
# edges = cv2.Canny(img, 100, 200)
# control = Image.fromarray(edges)

scales = [0.2, 0.4, 0.6, 0.8, 1.0]
for scale in scales:
    print(f'Scale {scale}: {"very loose" if scale<0.4 else "balanced" if scale<0.7 else "strict"}')
    # result = pipe(prompt, image=control, controlnet_conditioning_scale=scale)
print('\n0.7 = sweet spot for most use cases')


## Ex 2: Inpainting — Object Replacement


In [ ]:
# from diffusers import AutoPipelineForInpainting
# pipe = AutoPipelineForInpainting.from_pretrained(
#     'diffusers/stable-diffusion-xl-1.0-inpainting-0.1',
#     torch_dtype=torch.float16, variant='fp16'
# ).to('cuda')
# 
# result = pipe(
#     prompt='A modern leather armchair, warm lighting',
#     image=room_image, mask_image=mask,
#     strength=0.99, guidance_scale=8.0,
#     padding_mask_crop=32,
# ).images[0]
print('strength=0.99 (not 1.0!) for max creativity')
print('padding_mask_crop=32 improves small masked regions')


## Ex 3: Multi-ControlNet


In [ ]:
# controlnets = [
#     ControlNetModel.from_pretrained('diffusers/controlnet-depth-sdxl-1.0-small'),
#     ControlNetModel.from_pretrained('diffusers/controlnet-canny-sdxl-1.0'),
# ]
# pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
#     'stabilityai/stable-diffusion-xl-base-1.0',
#     controlnet=controlnets
# ).to('cuda')
# result = pipe(
#     prompt='Modern building, sunset',
#     image=[depth_map, canny_edges],
#     controlnet_conditioning_scale=[0.8, 0.5],
#     control_guidance_end=[0.8, 0.5],
# ).images[0]
print('Multi-ControlNet: pass lists for images, scales, timing')
print('~70% of production requests use 2+ ControlNets')


## Ex 4: IP-Adapter Style Transfer


In [ ]:
# pipe.load_ip_adapter('h94/IP-Adapter', subfolder='sdxl_models',
#                      weight_name='ip-adapter_sdxl.bin')
# pipe.set_ip_adapter_scale(0.6)
# result = pipe(
#     prompt='Product photography, studio lighting',
#     ip_adapter_image=reference_style_image,
# ).images[0]
print('IP-Adapter: ~22M params, zero training')
print('Scale 0.6 = balanced text + image influence')
print('Per-block: down_block_2 (layout), up_block_0 (style)')


## Ex 5: Post-Processing Pipeline


In [ ]:
# The production pipeline (conceptual)
def production_pipeline(image):
    steps = [
        ('1. Generate', '1024x1024 at native resolution'),
        ('2. Hires Fix', 'Upscale 2x + denoise 0.3-0.5'),
        ('3. ADetailer', 'YOLO face detect → inpaint fix, denoise 0.3-0.4'),
        ('4. CodeFormer', 'Face restoration, fidelity=0.5'),
        ('5. Real-ESRGAN', 'Final 4x upscale, no denoising'),
    ]
    for step, desc in steps:
        print(f'{step}: {desc}')
    print('\nResult: ~100-200 publishable images/hour on RTX 4090')

production_pipeline(None)


## Ex 6: Regional Prompting Concept


In [ ]:
# Regional prompting prevents attribute confusion
print('Problem: "man in red, woman in blue" → colors swap!')
print()
print('Solution: Regional Prompting')
print('  ComfyUI: Attention Couple nodes + binary masks')
print('  A1111: Regional Prompter with BREAK separator')
print('  diffusers: ConditioningSetArea')
print()
print('Prompt weighting: (important word:1.5) boosts attention')
print('Range: 0.5-1.5 (beyond 1.5 = artifacts)')
print()
print('Attend-and-Excite: prevents forgotten subjects')
print('  from diffusers import StableDiffusionAttendAndExcitePipeline')


## Ex 7: India Cost Calculator


In [ ]:
use_cases = [
    ('Architectural render', 40000, 250, 'ControlNet depth/canny + LoRA'),
    ('Product photo', 5000, 50, 'Inpainting bg replace + ControlNet'),
    ('Fashion catalog (50 images)', 75000, 500, 'ControlNet pose + LoRA'),
    ('Interior design render', 15000, 100, 'ControlNet canny + prompt'),
    ('Wedding album (100 photos)', 50000, 2000, 'CodeFormer + Real-ESRGAN'),
]

print(f'{"Use Case":35s} {"Traditional":>12s} {"AI":>8s} {"Savings":>8s}')
print('-' * 68)
for name, trad, ai, tech in use_cases:
    savings = (1 - ai/trad) * 100
    print(f'{name:35s} ₹{trad:>10,} ₹{ai:>6,} {savings:>6.0f}%')

print('\nIndian companies: Dresma (9M+ images), ZYNG AI, Ecom Model Studio')
print('Key gap: Indic text rendering — use PIL/Pango compositing')


## Ex 8: ComfyUI API Automation


In [ ]:
import json

# ComfyUI workflow automation pattern
workflow = {
    '3': {'class_type': 'KSampler', 'inputs': {
        'seed': 42, 'steps': 25, 'cfg': 7.5,
        'sampler_name': 'dpmpp_2m', 'scheduler': 'karras',
    }},
    '6': {'class_type': 'CLIPTextEncode', 'inputs': {
        'text': 'Modern Indian building, golden hour lighting'
    }},
}

print('ComfyUI API Pattern:')
print('1. Build workflow visually in ComfyUI')
print('2. Save as API format JSON')
print('3. Modify parameters by node ID:')
print(json.dumps(workflow, indent=2))
print('4. POST to http://127.0.0.1:8188/prompt')
print('5. Track via WebSocket, retrieve from /history')


---
## Recap
Controlled generation = ControlNet (structure) + LoRA (style) + IP-Adapter (reference) + Inpainting (editing) + Post-processing (ADetailer→CodeFormer→Real-ESRGAN) + ComfyUI (automation).

Multi-ControlNet and Union models for combined signals. Regional prompting for multi-subject scenes. Indian opportunities: architecture (99% savings), fashion (99%), e-commerce (85-97%).
